In [12]:
# Resstock model download, and ami optimization 

"""
Created on Sep 11 16:00:00 2025

@author: Tanushree Charan

"""

'\nCreated on Sep 11 16:00:00 2025\n\n@author: Tanushree Charan\n\n'

In [13]:
pip install awscli


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import subprocess
import zipfile
import os
import shutil
import json
import pandas as pd
from pathlib import Path

In [10]:
# Read list of all building IDs

# path to metadata parquet

current_path = Path(os.getcwd())
data_path = current_path / '..'/ 'data' 
buildstock_metadata_path = data_path / 'NY_baseline_metadata_and_annual_results.xlsx'

# Read the "select_counties" sheet
df = pd.read_excel(buildstock_metadata_path, sheet_name="select_counties")

# Number of rows in the DataFrame
num_rows = len(df)
print("Number of rows:", num_rows)

building_ids_metadata = df["bldg_id"]

# Save building IDs to CSV
output_path = data_path / "base_files" / "building_ids_metadata.csv"
building_ids_metadata.to_csv(output_path, index=False)

print("bldg_id:", len(building_ids_metadata))
print(building_ids_metadata)
print('done')

Number of rows: 607
bldg_id: 607
0         510
1        1655
2        1700
3        3424
4        6034
        ...  
602    546088
603    547188
604    547247
605    547426
606    548817
Name: bldg_id, Length: 607, dtype: int64
done


In [11]:
# List of building IDs to download
#building_ids = [140567, 253852, 379341, 335998, 43442, 491645, 533465, 275002, 295328, 302371, 303300, 303512, 310716, 269317, 273946, 329817, 350503, 364293, 65865, 421718, 438492, 444781, 115980, 88748, 458439]

building_ids = [140567, 253852, 379341]

# Base S3 path for ResStock 2024 release
s3_base = (
    "s3://oedi-data-lake/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2024/"
    "resstock_tmy3_release_2/model_and_schedule_files/"
    "building_energy_models/upgrade=0/"
)

# base files
base_files = Path("../data/base_files")
measures_og = Path(base_files / "measures")
workflow_og = Path(base_files / "workflow_resstock.osw")
# weather file
weather_file = "Ithaca_2024.epw"
weather_og = Path(base_files / "weather" / weather_file)

# Output directories
extract_dir = Path("../data/resstock_models")
extract_dir.mkdir(exist_ok=True)

# Loop through each ID and download + unzip
for bid in building_ids:
    
    padded_id = f"{bid:07d}"
    filename = f"bldg{padded_id}-up00.zip"
    s3_path = s3_base + filename

    local_extract_path = extract_dir / f"bldg{padded_id}"
    zip_file = local_extract_path / filename
    # make models folder
    local_extract_path_models = extract_dir / f"bldg{padded_id}" / "models"

    ### DOWNLOAD MODELS
    print(f"Downloading {filename}...")
    try:
        subprocess.run(
            ["aws", "s3", "cp", s3_path, str(zip_file), "--no-sign-request"],
            check=True
        )
    except subprocess.CalledProcessError as e:
        print(f"Failed to download {filename}: {e}")
        continue

    print(f"Unzipping {filename} to folder {local_extract_path}...")
    try:
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            local_extract_path.mkdir(exist_ok=True)
            local_extract_path_models.mkdir(exist_ok=True)

            zip_ref.extractall(local_extract_path_models)
    except zipfile.BadZipFile as e:
        print(f"Bad ZIP file: {filename}: {e}")
        continue
        
    ### WEATHER
    # create weather folder if it doesn't exist
    weather_new = os.path.join(local_extract_path, 'weather')
    os.makedirs(weather_new, exist_ok=True)  # Create folder using filename if it doesn't exist
    shutil.copy(weather_og, weather_new)

    ### MEASURES
    # copy default measures
    new_measure_path = os.path.join(local_extract_path, 'measures')

    # Check if the folder already exists
    if os.path.exists(new_measure_path):
        # If the folder already exists, delete it
        try:
            shutil.rmtree(new_measure_path)  # Delete the existing folder
            print(f"Deleted existing folder at: {new_measure_path}")
        except Exception as e:
            print(f"Error deleting folder: {e}")
            raise

    # Copy the contents of the measure_path to new_measure_path
    try:
        shutil.copytree(measures_og, new_measure_path)
        print("All folders and files copied successfully.")
    except Exception as e:
        print(f"Error copying files: {e}")


    #### OSW FILE
    new_osw_path = os.path.join(local_extract_path, 'workflow_resstock.osw')
    shutil.copy(workflow_og, new_osw_path)

    with open(new_osw_path, 'r') as osw:
        data = json.load(osw)
        data['seed_file'] =  "in.osm"
        data['weather_file'] = f"{weather_file}"

    with open(new_osw_path, 'w') as osw:
        json.dump(data, osw, indent=2)


    # Optional: remove the ZIP file after extraction
    #zip_file.unlink()

print("Done.")


download: s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/model_and_schedule_files/building_energy_models/upgrade=0/bldg0140567-up00.zip to ../data/resstock_models/bldg0140567/bldg0140567-up00.zip
Unzipping bldg0140567-up00.zip to folder ../data/resstock_models/bldg0140567...
Deleted existing folder at: ../data/resstock_models/bldg0140567/measures
All folders and files copied successfully.
download: s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/model_and_schedule_files/building_energy_models/upgrade=0/bldg0253852-up00.zip to ../data/resstock_models/bldg0253852/bldg0253852-up00.zip
Unzipping bldg0253852-up00.zip to folder ../data/resstock_models/bldg0253852...
Deleted existing folder at: ../data/resstock_models/bldg0253852/measures
All folders and files copied successfully.
download: s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-

In [12]:
# RUN OPENSTUDIO MODELS 1 at a time 

# Windows
#openstudio_path="C:/openstudio-3.9.0/bin/openstudio.exe"

# Mac
openstudio_path="/Applications/OpenStudio-3.9.0/bin/openstudio"


# Check if OpenStudio is installed and accessible
try:
    result = subprocess.run([openstudio_path, "openstudio_version"], capture_output=True, text=True, check=True)
    print(f"OpenStudio is installed at {openstudio_path}.")

except subprocess.CalledProcessError:
    print(f"OpenStudio is not installed or not accessible at {openstudio_path}.")

# Loop through each ID and download + unzip
for bid in building_ids:
    
    padded_id = f"{bid:07d}"
    filename = f"bldg{padded_id}-up00.zip"
    
    local_extract_path = extract_dir / f"bldg{padded_id}"
    
    osw = Path(os.path.join(local_extract_path, 'workflow_resstock.osw'))
    
    # Run the OpenStudio command
    # TO DO: Run 15 min timestep
    result = subprocess.run([openstudio_path, "run", "-w", osw], capture_output=True, text=True)
    print(result)
    # Assert that there are no errors
    assert result.returncode == 0, f"OpenStudio run failed for {bid}: {result.stderr}"
    assert result.stderr == "", f"Unexpected stderr output for {bid}: {result.stderr}"

OpenStudio is installed at /Applications/OpenStudio-3.9.0/bin/openstudio.
CompletedProcess(args=['/Applications/OpenStudio-3.9.0/bin/openstudio', 'run', '-w', PosixPath('../data/resstock_models/bldg0140567/workflow_resstock.osw')], returncode=0, stdout='', stderr='')
CompletedProcess(args=['/Applications/OpenStudio-3.9.0/bin/openstudio', 'run', '-w', PosixPath('../data/resstock_models/bldg0253852/workflow_resstock.osw')], returncode=0, stdout='', stderr='')
CompletedProcess(args=['/Applications/OpenStudio-3.9.0/bin/openstudio', 'run', '-w', PosixPath('../data/resstock_models/bldg0379341/workflow_resstock.osw')], returncode=0, stdout='', stderr='')


### RUN MODELS IN PARALLEL

In [13]:
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Windows
#openstudio_path="C:/openstudio-3.9.0/bin/openstudio.exe"

# Mac
openstudio_path="/Applications/OpenStudio-3.9.0/bin/openstudio"

# Quick check
subprocess.run([openstudio_path, "openstudio_version"], check=True, capture_output=True, text=True)

MAX_PARALLEL = 2
building_ids = [140567, 253852, 379341]
extract_dir = Path("../data/resstock_models")

def run_one(bid: int):
    padded = f"{bid:07d}"
    osw = Path(extract_dir) / f"bldg{padded}" / "workflow_resstock.osw"
    if not osw.exists():
        return bid, 1, f"Missing OSW: {osw}"
    r = subprocess.run([openstudio_path, "run", "-w", str(osw)], capture_output=True, text=True)
    return bid, r.returncode, r.stderr.strip()

print(f"Running {len(building_ids)} models (up to {MAX_PARALLEL} at once)...")
errors = []

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    # start all jobs at once (up to MAX_PARALLEL run concurrently)
    futures = [ex.submit(run_one, bid) for bid in building_ids]

    # handle results as each job finishes
    for i, fut in enumerate(as_completed(futures), 1):
        bid, rc, err = fut.result()        # what run_one() returns
        ok = (rc == 0 and err == "")       # success if no error and rc==0
        print(f"[{i}/{len(building_ids)}] bldg {bid}: {'OK' if ok else 'FAIL'}")
        if not ok:
            errors.append((bid, rc, err))

# after all jobs finish, fail the cell if any failed
if errors:
    msg = " | ".join([f"{bid}: rc={rc}, err={err[:120]}" for bid, rc, err in errors])
    raise AssertionError(f"OpenStudio failed for {len(errors)} model(s): {msg}")

print("All done")


Running 3 models (up to 2 at once)...
[1/3] bldg 253852: OK
[2/3] bldg 140567: OK
[3/3] bldg 379341: OK
All done


In [44]:
# RESSTOCK DATA PREPARATION FOR OPTIMIZATION
# CONVERT RESULT FILES TO PARQUET AND STORE ANNUAL RESULTS



# cols = {"id": "bldg_id",
#         "time": "timestamp",
#         "ts_elec": "electricity.total.energy_consumption",
#         "annl_elec": "out.electricity.total.energy_consumption"}


# Save in data/parquet Folder
parquet_folder = Path("../data/parquet")
parquet_folder.mkdir(exist_ok=True)

frames = []
target_cols = ["Date/Time", "EMS:EMS_ElectricityFacility_kWh [](TimeStep)"]

# find eplusout.csv file
for bldg_folder in extract_dir.iterdir():
    if bldg_folder.is_dir():
        csv_path = bldg_folder.rglob("eplusout.csv")
        csv_path = next(csv_path, None)  # get first match
        if csv_path:
            bldg_id = bldg_folder.name
            try:
                df = pd.read_csv(csv_path, usecols=target_cols)
                
            except ValueError as e:
                print(f"Required columns not found in {csv_path}: {e}")
                continue
                
            # save to parquet
            df = df.rename(
                    columns={
                        "Date/Time": "time",
                        "EMS:EMS_ElectricityFacility_kWh [](TimeStep)": "ts_elec"
                    })
            
            df["bldg_id"] = int(bldg_id.replace("bldg",""))
            # CURRENTLY TIME STEP IS 15, AGGREGATE TO 1 HOUR IF NEEDED                          
            # Step 1: Clean up spaces
            df["time"] = df["time"].str.strip().str.replace("  ", " ")
            
            # Step 2: Fix special "24:00:00" to "00:00:00"
            df["time"] = df["time"].str.replace("24:00:00", "00:00:00")
            
            # Step 3: Add the year 2024 in front
            df["time"] = "2024-" + df["time"]
            
            # Step 4: Convert to datetime
            df["time"] = pd.to_datetime(df["time"], format="%Y-%m/%d %H:%M:%S")
            frames.append(df)
        else:
            print(f"No eplusout.csv found in {bldg_folder}")


if frames:
    # TIMESERIES PARQUET FILE
    all_df = pd.concat(frames, ignore_index=True)
    all_df = all_df.sort_values(["bldg_id", "time"])
    # TIMESERIES 1 HOUR PARQUET FILE
    all_df_hour = all_df.set_index("time").groupby("bldg_id")["ts_elec"].resample("h").sum().reset_index()

    
    out_path = parquet_folder / "resstock_timeseries_all_15.parquet"
    all_df.to_csv(parquet_folder / "resstock_timeseries_all_15.csv", index=False)
    all_df.to_parquet(out_path, index=False)

    all_df_hour.to_csv(parquet_folder / "resstock_timeseries_all_hour.csv", index=False)
    all_df.to_parquet(parquet_folder / "resstock_timeseries_all_hour.parquet", index=False)
    
    print(f"Saved {out_path}")


    # ANNUAL PARQUET FILE
    df_anl = all_df.groupby("bldg_id", as_index=False)['ts_elec'].sum()
    df_anl = df_anl.rename(columns={"ts_elec": "annl_elec"})
    out_path_anl = parquet_folder / "resstock_timeseries_annual.parquet"
    df_anl.to_parquet(out_path_anl, index=False)
    df_anl.to_csv(parquet_folder / "resstock_timeseries_annual.csv")
    print(f"Saved {out_path_anl}")

else:
    print("No timeseries found.")


Saved ../data/parquet/resstock_timeseries_all_15.parquet
Saved ../data/parquet/resstock_timeseries_annual.parquet


In [46]:
# AMI DATA PREPARATION FOR OPTIMIZATION

# REQUIRED

# ami_cols = {"id": ami_idx,
#             "id_metadata": ami_meta_idx,
#             "time": "timestamp",
#             "ts_elec": "value",} 
# ami_cols = {"id": "SERVIDEPOINTID",
#            "time": "ENDTIME_EST",
#            "ts_elec": "INTERVAL_READ"}

# CURRENT FORMAT

# SERVICEPOINTID	CHANNELNUMBER	ENDTIME_EST	INTERVAL_READ
# 6000973858	1	6/13/23 14:00	0.057
# 6000973858	1	6/13/23 14:15	0.051


# Path to AMI data

parquet_folder = Path("../data/parquet")
ami = Path("/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/AMI/NREL/consumption")
ami_ind = ami / "ch1_4301002_individual.csv"

df = pd.read_csv(ami_ind)
# parse date time 
df["ENDTIME_EST"] = pd.to_datetime(df["ENDTIME_EST"])
df_2024 = df[df["ENDTIME_EST"].dt.year == 2024]

# total rows
print("Total rows:", len(df_2024))

# total unique service IDs
unique_ids = df_2024["SERVICEPOINTID"].nunique()
print("Total unique service IDs:", unique_ids)

# Define the keys that must be unique
keys = ["SERVICEPOINTID", "ENDTIME_EST"]

# Average duplicates (singles stay the same), write back into INTERVAL_READ
df_2024 = (
    df_2024
    .groupby(keys, as_index=False)["INTERVAL_READ"]
    .mean()
)

# REMOVE FEB 29 FROM AMI DATA SET
df_2024 = df_2024[df_2024["ENDTIME_EST"].dt.date != pd.to_datetime("2024-02-29").date()]


# make hourly data
df_2024_hour = (
    df_2024.set_index("ENDTIME_EST")
           .groupby("SERVICEPOINTID")["INTERVAL_READ"]
           .resample("h").sum()
           .reset_index()
)

out_path_15 = parquet_folder / "ami_ind_data_15.parquet"
df_2024.to_parquet(out_path_15, index=False)
df_2024.to_csv(parquet_folder / "ami_ind_data_15.csv", index=False)

df_2024_hour.to_parquet(parquet_folder / "ami_ind_data_hour.parquet", index=False)
df_2024_hour.to_csv(parquet_folder / "ami_ind_data_hour.csv", index=False)

print("Saved:", out_path_15)
print(df_2024.head())

# 6000974613	1	12/31/24 23:45	0.359
# 6000974626	1	1/1/24 0:00	0.144

Total rows: 21380228
Total unique service IDs: 670
Saved: ../data/parquet/ami_ind_data_15.parquet
   SERVICEPOINTID         ENDTIME_EST  INTERVAL_READ
0      6000973858 2024-01-01 00:00:00          0.095
1      6000973858 2024-01-01 00:15:00          0.112
2      6000973858 2024-01-01 00:30:00          0.118
3      6000973858 2024-01-01 00:45:00          0.069
4      6000973858 2024-01-01 01:00:00          0.081


In [39]:
## CHECK FOR DUPLICATES IN AMI

# number of service IDs with more than 1 timeseries (i.e. multiple CHANNELNUMBERs)
service_id_counts = df_2024.groupby("SERVICEPOINTID")["CHANNELNUMBER"].nunique()
multi_ts_ids = (service_id_counts > 1).sum()
print("Number of service IDs with >1 timeseries:", multi_ts_ids)

# --- Check for duplicates ---

keys = ["SERVICEPOINTID", "ENDTIME_EST"]

# Find duplicates
dup_mask = df_2024.duplicated(subset=keys, keep=False)

# Filter duplicates
df_dupes = df_2024[dup_mask].copy()

# Count unique service IDs with duplicates
unique_sid_count = df_dupes["SERVICEPOINTID"].nunique()

print("Number of duplicate rows:", len(df_dupes))
print("Number of service IDs with duplicate rows:", unique_sid_count)

# Save duplicates to CSV
out_csv = parquet_folder / "ami_duplicates_2024.csv"
df_dupes.to_csv(out_csv, index=False)
print("Saved duplicates to:", out_csv)

# --- Average duplicates (collapse to one row per SERVICEPOINTID + ENDTIME_EST) ---
df_2024_avg = (
    df_2024
    .groupby(keys, as_index=False)["INTERVAL_READ"]
    .mean()
    .rename(columns={"INTERVAL_READ": "INTERVAL_READ_avg"})
)

# Sanity check
print("Before rows:", len(df_2024))
print("After rows :", len(df_2024_avg))
print(df_2024_avg.head())


Number of duplicate rows: 5358
Number of service IDs with duplicate rows: 670
Saved duplicates to: ../data/parquet/ami_duplicates_2024.csv
Before rows: 21380228
After rows : 21377549
   SERVICEPOINTID         ENDTIME_EST  INTERVAL_READ_avg
0      6000973858 2024-01-01 00:00:00              0.095
1      6000973858 2024-01-01 00:15:00              0.112
2      6000973858 2024-01-01 00:30:00              0.118
3      6000973858 2024-01-01 00:45:00              0.069
4      6000973858 2024-01-01 01:00:00              0.081


##### CAVEATS AND ASSUMPTIONS
##### The models in Resstock resstock_tmy3_release_2 are not for a leap year. The schedule files do not have a schedule for Feb 29 
##### If run for 2024 in the setRunPeriod measure - throws an error
##### these are run for 2023 in setRunPeriod - BUT are using Ithaca 2024 .epw weather file
##### When running optimization - AMI data for Feb 29 would have to be deleted
##### Only Resstock right now


### Selected counties: 
###792 unique service point IDs, 154 commercial
###ResStock Models: 

###New York State : 33791 models

###Tompkins County : 174 models

###Tompkins and neighboring counties with similar geographic landscape and building construction:

###Tompkins Cortland, Tioga, Chemung, Schuyler, Seneca, Cayuga: 607 models​


In [ ]:
# LOAD PARQUET FILE AND SEE RESULTS

# path to the parquet file AWS
path = Path("/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/NY_parquet_results")
ind_parq = path / "10000-0.parquet"
anl_parq = path / "NY_baseline_metadata_and_annual_results.parquet"

# path to the parquet file pgh
path_pgh = Path(path, "pgh")
ind_parq_pgh = path_pgh / "sample_resstock_ts.parquet"
anl_parq_pgh = path_pgh / "resstock_metro_pgh_results.parquet"

df_1 = pd.read_parquet(ind_parq_pgh)
df_1.to_csv(path_pgh / "ind_parq_pgh.csv", index=False)
print(df_1.shape)         # rows, cols
print(df_1.columns.tolist())

df_2 = pd.read_parquet(anl_parq_pgh)
df_2.to_csv(path_pgh / "anl_parq_pgh.csv", index=False)
print(df_2.shape)         # rows, cols
#print(df_2.columns.tolist())

# df_3 = pd.read_parquet(ind_parq)
# df_3.to_csv(path / "ind_parq.csv", index=False)
# print(df.shape)         # rows, cols
# print(df.columns.tolist())

# df_4 = pd.read_parquet(anl_parq)
# df_4.to_csv(path / "anl_parq.csv", index=False)
# print(df_4.shape)         # rows, cols
# print(df_4.columns.tolist())

print('done')

In [ ]:
# Reference Script

# https://github.com/NREL/buildstock-ami-mapping/blob/e2c-pgh-testing/notebooks/run_model_e2c_pgh.ipynb
# Required fields: 
# cols = {"id": "bldg_id",
#         "time": "timestamp",
#         "ts_elec": "electricity.total.energy_consumption",
#         "annl_elec": "out.electricity.total.energy_consumption"}

# out.electricity.total.energy_consumption.kwh